In [21]:
import pandas as pd
import os
import zipfile
from pathlib import Path
import numpy as np
import re
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from functools import reduce
import gc

* Variables and Functions setup

In [ ]:
zip_folder = 'raw_data'
data_folder = 'extracted_data'
survey_year_79 = [y for y in range(1979,2023) if y not in [1995, 1997, 1999, 2001, 2003, 2005, 2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021]]
survey_year_97 = [y for y in range(1997,2024) if y not in [2012,2014,2016,2018,2020,2022]]
race_mapping = {
    3: 'White', 5: 'White', 6: 'White', 7: 'White', 11: 'White', 12: 'White', 
    22: 'White', 23: 'White', 24: 'White', 25: 'White', 27: 'White', 29: 'White', 
    1: 'Black',
    15: 'Hispanic', 16: 'Hispanic', 17: 'Hispanic', 18: 'Hispanic', 19: 'Hispanic', 
    20: 'Hispanic', 21: 'Hispanic',
    2: 'Asian/PI', 4: 'Asian/PI', 8: 'Asian/PI', 10: 'Asian/PI', 13: 'Asian/PI', 
    14: 'Asian/PI', 26: 'Asian/PI',
    0: 'Native/Other', 9: 'Native/Other', 28: 'Native/Other' 
}
sex_mapping = {1:"Male", 2:"Female"}

In [ ]:
def create_rnum_dict(file_path):
    rnum_to_var = {}
    pattern = re.compile(r"^([A-Za-z0-9]+\.\d{2}[A-Za-z0-9]*)\s+\[(.*?)\]\s+Survey Year:\s*(.*)")
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        for line in file:
            match = pattern.search(line.strip())
            if match:
                rnum = re.sub(r'[^a-zA-Z0-9]', '', match.group(1))
                var_name = match.group(2)
                survey_year_79 = match.group(3).strip()
                if re.match(r"^\d{4}$", survey_year_79):
                    var_name = f"{var_name}_{survey_year_79}"
                rnum_to_var[rnum] = var_name     
    return rnum_to_var

# 79 rename dictionary
def rename_columns(col):
    if 'EMPLOYERS_ALL_' in col:
        clean_name = col.replace('EMPLOYERS_ALL_', '')
        if 'UID' in col:
            return clean_name
        match_gap = re.match(r'(.+)_(\d{4})\.(\d+)\.(\d+)$',clean_name)
        if match_gap:
            var, year, job_num, gap_num = match_gap.groups()
            return f"{var}.{job_num}_{gap_num}_{year}"
        match_job = re.match(r'(.+)_(\d{4})\.(\d+)$',clean_name)
        if match_job:
            var, year, job_num = match_job.groups()
            return f"{var}.{job_num}_{year}"
        return clean_name
    elif 'CASEID' in col:
        return 'CASEID'
    elif 'HGCREV' in col:
        parts = col.split('_')
        return f"{parts[0][:-2]}_{parts[1]}"
    elif 'FAM-30_1_1979' in col:
        return 'race'
    elif 'SAMPLE_SEX_1979' in col:
        return 'sex'
    return col

* Occupation major grouping maping

In [ ]:
occ70_dict = {}
for x in range(0, 196):
    occ70_dict[x] = "Professional"
for x in range(201, 246):
    occ70_dict[x] = "Managers"
for x in range(260,281):
    occ70_dict[x] = "Sales"
for x in range(301,396):
    occ70_dict[x] = "Clerical"
for x in range(401,581):
    occ70_dict[x] = "Craftsmen"
# for x in range(601,727):
#     occ70_dict[x] = "Operatives"
# for x in range(740,786):
#     occ70_dict[x] = "Laborers"
for x in range(601,786):
    occ70_dict[x] = "Operatives and Laborers"
for x in range(901,985):
    occ70_dict[x] = "Service"

    
occ00_dict = {}
# Professional
for x in [range(62,71),80,82,93,range(100,352),353,354,361,362,363,462,493,601,903,904]:
    if type(x) is int:
        occ00_dict[x] = "Professional"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Professional"
# Managers
for x in [range(0,54),56,72,73,83,range(84,92),94,95,931]:
    if type(x) is int:
        occ00_dict[x] = "Managers"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Managers"
# Sales
for x in [range(475,493),range(494,497)]:
    if type(x) is int:
        occ00_dict[x] = "Sales"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Sales"
# Clerical
for x in [54,60,81,472,474,500,range(510,556),range(556,594)]:
    if type(x) is int:
        occ00_dict[x] = "Clerical"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Clerical"
# Craftsmen
for x in [352,441,range(621,626),range(630,637),range(642,654),range(666,672),674,676,range(700,761),range(762,771),774,780,803,806,813,816,range(823,826),
          833,835,844,845,range(850,853),range(860,864),875,876,891,900,920,941,range(951,957)]:
    if type(x) is int:
        occ00_dict[x] = "Craftsmen"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Craftsmen"
# # Operatives
# for x in [range(501,504),640,range(680,692),694,range(771,774),775,range(781,803),804,range(810,813),814,815,range(820,823),range(826,833),834,range(836,843),846,range(853,856),
#           range(864,875),range(880,891),range(892,895),896,range(911,916),923,933,935,936,942,950,960,964,965,973]:
#     if type(x) is int:
#         occ00_dict[x] = "Operatives"
#     elif type(x) is range:
#         for n in x:
#             occ00_dict[n] = "Operatives"
# # Laborers
# for x in [421,425,434,435,600,range(602,614),626,660,672,673,675,692,693,761,895,range(961,964),972,974,975]:
#     if type(x) is int:
#         occ00_dict[x] = "Laborers"
#     elif type(x) is range:
#         for n in x:
#             occ00_dict[n] = "Laborers"
# Operatives and Laborers
for x in [range(501,504),640,range(680,692),694,range(771,774),775,range(781,803),804,range(810,813),814,815,range(820,823),range(826,833),834,range(836,843),846,range(853,856),
          range(864,875),range(880,891),range(892,895),896,range(911,916),923,933,935,936,942,950,960,964,965,973,421,425,434,435,600,range(602,614),626,660,672,673,675,692
          ,693,761,895,range(961,964),972,974,975]:
    if type(x) is int:
        occ00_dict[x] = "Operatives and Laborers"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Operatives and Laborers"
# Service
for x in [360,364,365,range(370,421),range(422,425),430,432,range(442,462),464,465,934]:
    if type(x) is int:
        occ00_dict[x] = "Service"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Service"

* Extracting files from zip files in raw_data folder, using zip_extract.py
  

* Generating csv map

In [ ]:
# Create diction of CPI, base year as 2020
df_cpi = pd.read_csv('cpi.csv')
df_cpi['observation_date'] = df_cpi['observation_date'].str[:4].astype(int)
base = df_cpi.loc[df_cpi['observation_date'] == 2020, 'CPIAUCSL'].values[0]
df_cpi['ratio'] = df_cpi['CPIAUCSL'] / base
cpi_map = df_cpi.set_index('observation_date')['ratio'].to_dict()

### Experience

In [ ]:
df79_exp = pd.read_csv(os.path.join(data_folder, '79-experience.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, '79-experience.cdb'))
df79_exp = df79_exp.rename(columns = rename_dict)
df79_exp = df79_exp.rename(columns = rename_columns)
df79_exp = df79_exp.replace([-1, -2, -3, -4,-5], np.nan)

# Start Week
start_cols = [col for col in df79_exp.columns if col.startswith("STARTWEEK.")]
df79_startweek_long = df79_exp[["CASEID"] + start_cols].melt(id_vars="CASEID",value_vars=start_cols,var_name="variable",value_name="Startweek")
df79_startweek_long.dropna(subset=['Startweek'],inplace=True)
df79_startweek_long[["job_num","year"]] = df79_startweek_long["variable"].str.extract(r"STARTWEEK\.(\d+)_(\d{4})")
df79_startweek_long["year"] = pd.to_numeric(df79_startweek_long["year"])
df79_startweek_long["job_num"] = pd.to_numeric(df79_startweek_long["job_num"])
df79_startweek_long = df79_startweek_long.drop(columns="variable")

# Stop Week
stop_cols = [col for col in df79_exp.columns if col.startswith("STOPWEEK.")]
df79_stopweek_long = df79_exp[["CASEID"] + stop_cols].melt(id_vars="CASEID",value_vars=stop_cols,var_name="variable",value_name="Stopweek")
df79_stopweek_long.dropna(subset=['Stopweek'],inplace=True)
df79_stopweek_long[["job_num","year"]] = df79_stopweek_long["variable"].str.extract(r"STOPWEEK\.(\d+)_(\d{4})")
df79_stopweek_long["year"] = pd.to_numeric(df79_stopweek_long["year"])
df79_stopweek_long["job_num"] = pd.to_numeric(df79_stopweek_long["job_num"])
df79_stopweek_long = df79_stopweek_long.drop(columns="variable")

# Merging
df79_jobweeks_long = pd.merge(df79_startweek_long,df79_stopweek_long,on=["CASEID", "year", "job_num"],how="inner")
df79_jobweeks_long = df79_jobweeks_long.sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)
df79_jobweeks_long = df79_jobweeks_long[df79_jobweeks_long['Stopweek'] >= df79_jobweeks_long['Startweek']].copy()
df79_jobweeks_long['segment'] = df79_jobweeks_long['Stopweek'] - df79_jobweeks_long['Startweek']

# Generating Experiences
df79_exp = df79_jobweeks_long[["CASEID", "job_num", "year", "Startweek", "segment"]].copy()
df79_exp = df79_exp.sort_values(["CASEID", "year", "Startweek", "job_num"])
df79_exp["experience"] = df79_exp.groupby("CASEID")["segment"].cumsum()
df79_exp = df79_exp[["CASEID", "job_num", "year", "experience"]].reset_index(drop=True)

# Minus Within Job Gap
df79_gap = pd.read_csv(os.path.join(data_folder, '79-job_gap.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, '79-job_gap.cdb'))
df79_gap = df79_gap.rename(columns = rename_dict)
df79_gap = df79_gap.rename(columns = rename_columns)
df79_gap = df79_gap.replace([-1, -2, -3, -4,-5], np.nan)

gap_cols = [c for c in df79_gap.columns if c.startswith(("PERSTAR",'PERSTOP'))]
df79_gap_long = df79_gap[['CASEID']+gap_cols].melt(id_vars = 'CASEID', value_vars = gap_cols, var_name = 'variable')
df79_gap_long.dropna(subset=['value'],inplace=True)
df79_gap_long[["gap_var", "job_num", "gap_num", "year"]] = df79_gap_long["variable"].str.extract(r"(PERSTAR|PERSTOP).(\d+)_(\d+)_(\d{4})")
df79_gap_long["job_num"] = pd.to_numeric(df79_gap_long["job_num"])
df79_gap_long["gap_num"] = pd.to_numeric(df79_gap_long["gap_num"])
df79_gap_long["year"] = pd.to_numeric(df79_gap_long["year"])
df79_gap_long = df79_gap_long.drop(columns="variable")
df79_gap_long = df79_gap_long.pivot_table(index=["CASEID", "year", "job_num", "gap_num"],columns="gap_var",values="value",aggfunc="first").reset_index()
df79_gap_long['duration'] = df79_gap_long['PERSTOP'] - df79_gap_long['PERSTAR']
df79_gap_sum = df79_gap_long.groupby(['CASEID','year','job_num'],as_index=False)['duration'].sum()

# Merging with exp df
df79_exp = df79_exp.merge(df79_gap_sum,on=['CASEID','year','job_num'],how='left')
df79_exp['duration'] = df79_exp['duration'].fillna(0)
df79_exp['experience'] = df79_exp['experience'] - df79_exp['duration']
df79_exp['experience'] = df79_exp['experience'] / 52
df79_exp = df79_exp.drop(columns='duration')

# Save memory spaces
del df79_startweek_long, df79_stopweek_long, df79_jobweeks_long, df79_gap_long, df79_gap_sum
gc.collect()

### NLSY79

In [ ]:
df79 = pd.read_csv(os.path.join(data_folder, 'nlsy79.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, 'nlsy79.cdb'))
df79 = df79.rename(columns = rename_dict)
df79 = df79.rename(columns=rename_columns)
df79 = df79.replace([-1, -2, -3, -4,-5], np.nan)

hour_cols = [col for col in df79.columns if col.startswith('HOURSWEEK.')]
wage_cols = [col for col in df79.columns if col.startswith('HRLY_WAGE.')]
occ_cols = [col for col in df79.columns if col.startswith('OCC.')]
asvab_cols = [c for c in df79.columns if c.startswith("ASVAB")]

for c in occ_cols:
    original = df79[c].copy()
    if int(c[-4:]) < 2002:
        df79[c] = original.map(occ70_dict)
    elif int(c[-4:]) == 2002:
        df79[c] = original.map(occ00_dict)
    elif int(c[-4:]) > 2002:
        df79[c] = (original / 10).map(occ00_dict)

for c in wage_cols:
    year = int(c[-4:])
    df79[c] = (df79[c] / 100)/ cpi_map[year]

In [ ]:
# # ASVAB three month or one year selection
# asvab_cols = [c for c in df79.columns if c.startswith("ASVAB")]

# # Comparing between 3 month grouping and one year grouping
# df_group = df79.copy()
# df_group["group_3month"] = (df_group["Q1-3_A~Y_1979"].astype(str) + "_Q" + (((df_group["Q1-3_A~M_1979"] - 1) // 3) + 1).astype(str))

# mean1 = (df_group.groupby("group_3month")[asvab_cols].mean().sort_index())
# mean2 = (df_group.groupby("Q1-3_A~Y_1979")[asvab_cols].mean().sort_index())

# ax = mean1.plot(figsize=(12, 6),linewidth=1)

# year_positions = {}
# for position, group_label in enumerate(mean1.index):
#     year = str(group_label).split("_")[0]
#     year_positions.setdefault(year, []).append(position)

# year_positions = {year: np.mean(positions) for year, positions in year_positions.items()}

# for column, line in zip(asvab_cols, ax.lines):
#     x_dots = []
#     y_dots = []
#     for year, value in mean2[column].items():
#         year_key = str(int(year))
#         if year_key in year_positions and pd.notna(value):
#             x_dots.append(year_positions[year_key])
#             y_dots.append(value)
#     ax.scatter(x_dots, y_dots, color=line.get_color(), edgecolor="black", linewidth=0.5,
#         s=45, zorder=5, label="_nolegend_")
# q1_positions = [i for i, label in enumerate(mean1.index) if str(label).upper().endswith("_Q1")]
# q1_labels = [str(mean1.index[i]).upper() for i in q1_positions]
# ax.set_xlabel("")
# ax.set_xticks(q1_positions)
# ax.set_xticklabels(q1_labels, rotation=0)
# ax.set_ylabel("Mean ASVAB score")
# ax.set_title("Three-Month ASVAB Means compared with One-Year Means - NLSY79 cohort")
# ax.get_legend().remove()
# ax.grid(alpha=0.3)

# plt.tight_layout()
# plt.show()

In [ ]:
# Normalization of ASVAB scores using Three-Year age group
df79 = pd.concat([df79,(df79["Q1-3_A~Y_1979"].astype(str)+ "_Q"+ (((df79["Q1-3_A~M_1979"] - 1) // 3) + 1).astype(str)).rename("group_3month")], axis=1)
std_cols = [f"{col}_std" for col in asvab_cols]
df79[std_cols] = df79.groupby("group_3month")[asvab_cols].transform(lambda x: (x - x.mean()) / x.std(ddof=1))
# df_79ability[std_cols].plot.density()

# Applying PCA analysis
valid_r = df79[std_cols].notna().all(axis=1)
valid_df = df79.loc[valid_r, std_cols]
pca = PCA(n_components=2)
pca_scores = pca.fit_transform(valid_df)
print("Variance explained by PC1:", f"{pca.explained_variance_ratio_[0]:.3f}")
df79["ASVAB"] = np.nan
df79.loc[valid_r, "ASVAB"] = pca_scores[:, 0]
# df79["ASVAB"] = df79["ASVAB"].rank(pct=True) * 100
# df79["ASVAB"] = (df79["ASVAB"] - df79["ASVAB"].mean()) / df79["ASVAB"].std(ddof=1)
df79.drop(columns=asvab_cols+std_cols+["group_3month"], inplace=True)
df79.dropna(subset=['ASVAB'], inplace=True)

In [ ]:
# Hoursweek
df79_hrsweek_long = df79.melt(id_vars=["CASEID"],value_vars=hour_cols,var_name="variable",value_name="HOURSWEEK").dropna(subset=["HOURSWEEK"])
df79_hrsweek_long[["job_num", "year"]] = df79_hrsweek_long["variable"].str.extract(r"HOURSWEEK\.(\d+)_(\d{4})")
df79_hrsweek_long["job_num"] = df79_hrsweek_long["job_num"].astype(int)
df79_hrsweek_long["year"] = df79_hrsweek_long["year"].astype(int)
df79_hrsweek_long = df79_hrsweek_long.drop(columns=["variable"]).sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)

# Wage
df79_wage_long = df79.melt(id_vars=["CASEID"],value_vars=wage_cols,var_name="variable",value_name="WAGE").dropna(subset=["WAGE"])
df79_wage_long[["job_num", "year"]] = df79_wage_long["variable"].str.extract(r"HRLY_WAGE\.(\d+)_(\d{4})")
df79_wage_long["job_num"] = df79_wage_long["job_num"].astype(int)
df79_wage_long["year"] = df79_wage_long["year"].astype(int)
df79_wage_long = df79_wage_long.drop(columns=["variable"]).sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)

# Occupation
df79_occ_long = df79.melt(id_vars=["CASEID"],value_vars=occ_cols,var_name="variable",value_name="OCC").dropna(subset=["OCC"])
df79_occ_long[["job_num", "year"]] = df79_occ_long["variable"].str.extract(r"OCC\.(\d+)_(\d{4})")
df79_occ_long["job_num"] = df79_occ_long["job_num"].astype(int)
df79_occ_long["year"] = df79_occ_long["year"].astype(int)
df79_occ_long = df79_occ_long.drop(columns=["variable"]).sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)

# Education
educ_cols = [c for c in df79.columns if c.startswith("HGCREV")]
educ_long = df79[["CASEID"] + educ_cols].melt(id_vars="CASEID",var_name="variable",value_name="HGCREV")
educ_long['year'] = educ_long['variable'].str.extract(r"(\d{4})$")[0].astype(int)
educ_long = educ_long.drop(columns='variable')

# Sample Weight
weight_cols = [c for c in df79.columns if c.startswith("SAMP")]
weight_long = df79[["CASEID"] + weight_cols].melt(id_vars="CASEID",var_name="variable",value_name="SampleWeight")
weight_long['year'] = weight_long['variable'].str.extract(r"(\d{4})$")[0].astype(int)
weight_long["SampleWeight"] = weight_long["SampleWeight"] / 100
weight_long = weight_long.drop(columns='variable')

# Age, Sex, Race, ASVAB
person_vars = df79[["CASEID", "sex", "race", "ASVAB",'Q1-3_A~Y_1979']].drop_duplicates(subset="CASEID")
person_vars["Q1-3_A~Y_1979"] = person_vars['Q1-3_A~Y_1979'] + 1900
person_vars['race'] = person_vars['race'].map(race_mapping)
person_vars['sex'] = person_vars['sex'].map(sex_mapping)

# Merge
df79_long = reduce(lambda left, right: pd.merge(left,right,on=["CASEID", "year", "job_num"],how="inner"),[df79_hrsweek_long,df79_wage_long,df79_occ_long,df79_exp])
df79_long = df79_long.merge(educ_long,on=["CASEID", "year"],how='left',validate='m:1')
df79_long = df79_long.merge(weight_long,on=["CASEID", "year"],how='left',validate='m:1')
df79_long = df79_long.merge(person_vars,on='CASEID',how='left',validate='m:1')
df79_long['Age'] = df79_long['year'] - df79_long['Q1-3_A~Y_1979']

# Additional adjustments
educ_map = df79_long[df79_long["job_num"] == 1].sort_values(["CASEID", "year"]).drop_duplicates(subset="CASEID", keep="first").set_index("CASEID")["HGCREV"]
df79_long['Educ'] = df79_long['CASEID'].map(educ_map)
df79_long = df79_long.drop(columns=['Q1-3_A~Y_1979','HGCREV'])

* Export to csv file

In [ ]:
df79_long.to_csv(os.path.join('cleaned', 'nlsy79_cleaned.csv'), index=False)

#### NLSY97

In [ ]:
df_97ability = pd.read_csv(os.path.join(data_folder, '97-ability.csv'))
abilty97_rename_dict = create_rnum_dict(os.path.join(data_folder, '97-ability.cdb'))
df_97ability = df_97ability.rename(columns = abilty97_rename_dict)
df_97ability = df_97ability.replace([-1, -2, -3, -4,-5], np.nan)
asvab_cols = [c for c in df_97ability.columns if c.startswith("ASVAB")]
subtests = ["GS", "AR", "WK", "PC", "NO", "CS", "MK", "MC"]
for test in subtests:
    pos = f"ASVAB_{test}_ABILITY_EST_POS"
    neg = f"ASVAB_{test}_ABILITY_EST_NEG"
    df_97ability[f"ASVAB_{test}"] = df_97ability[pos].combine_first(-df_97ability[neg])
    df_97ability = df_97ability.drop(columns=[pos,neg])
asvab_cols = [c for c in df_97ability.columns if c.startswith("ASVAB")]

# Comparing between 3 month grouping and one year grouping

df_97ability["group_3month"] = (df_97ability["KEY!BDATE_Y_1997"].astype(str) + "_Q" + (((df_97ability["KEY!BDATE_M_1997"] - 1) // 3) + 1).astype(str))

mean1 = (df_97ability.groupby("group_3month")[asvab_cols].mean().sort_index())
mean2 = (df_97ability.groupby("KEY!BDATE_Y_1997")[asvab_cols].mean().sort_index())

ax = mean1.plot(figsize=(12, 6),linewidth=1)

year_positions = {}
for position, group_label in enumerate(mean1.index):
    year = str(group_label).split("_")[0]
    year_positions.setdefault(year, []).append(position)

year_positions = {year: np.mean(positions) for year, positions in year_positions.items()}

for column, line in zip(asvab_cols, ax.lines):
    x_dots = []
    y_dots = []
    for year, value in mean2[column].items():
        year_key = str(int(year))
        if year_key in year_positions and pd.notna(value):
            x_dots.append(year_positions[year_key])
            y_dots.append(value)
    ax.scatter(x_dots, y_dots, color=line.get_color(), edgecolor="black", linewidth=0.5,
        s=45, zorder=5, label="_nolegend_")
q1_positions = [i for i, label in enumerate(mean1.index) if str(label).upper().endswith("_Q1")]

q1_labels = [str(mean1.index[i]).upper() for i in q1_positions]
ax.set_xlabel("")
ax.set_xticks(q1_positions)
ax.set_xticklabels(q1_labels, rotation=0)
ax.set_ylabel("Mean ASVAB score")
ax.set_title("Three-Month ASVAB Means compared with One-Year Mean - NLSY97 cohort")
ax.get_legend().remove()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Normalization of ASVAB scores using Three- age group
std_cols = [f"{col}_std" for col in asvab_cols]
df_97ability[std_cols] = (df_97ability.groupby("group_3month")[asvab_cols].transform(lambda x: (x - x.mean()) / x.std(ddof=1)))

# Applying PCA analysis
asvab_std_cols = [c for c in df_97ability.columns if c.startswith("ASVAB")]
valid_r = df_97ability[asvab_std_cols].notna().all(axis=1)
valid_df = df_97ability.loc[valid_r, asvab_std_cols]
scaler = StandardScaler()
scaled_asvab = scaler.fit_transform(valid_df)

pca = PCA(n_components=2)
pca_scores = pca.fit_transform(scaled_asvab)
print("Variance explained by PC1:", f"{pca.explained_variance_ratio_[0]:.3f}")
df_97ability['ASVAB'] = np.nan
df_97ability.loc[valid_r,'ASVAB'] = pca_scores[:, 0]
df_97ability['ASVAB'] = df_97ability['ASVAB'].rank(pct=True) * 100

df_97ability.drop(columns=asvab_cols+asvab_std_cols+["group_3month"], inplace=True)
df_97ability

### Generate job suffix match

In [ ]:
def rename_columns_97(col):
    if 'PUBID' in col:
        return 'PUBID'
    elif 'YEMP' in col:
        clean_name = col.replace('YEMP_', '')
        if '~' in clean_name:
            new_name = clean_name.replace('~','_')
            return new_name
        return clean_name
    return col

def clean_97(df):
    df = df.rename(columns=rename_columns_97).copy()
    month_cols = [c for c in df.columns if c.startswith("STARTDATE.") and "_M_" in c]
    for c in month_cols:
        year_col = c.replace("_M_", "_Y_", 1)
        date_col = c.replace("_M_", "_", 1)
        if year_col in df.columns:
            month = pd.to_numeric(df[c], errors="coerce")
            year = pd.to_numeric(df[year_col], errors="coerce")
            valid = month.between(1, 12) & year.gt(0)
            df[date_col] = year.mul(100).add(month).where(valid).astype("Int64")
    return df

In [62]:
df97_id = pd.read_csv(os.path.join(data_folder, '97-employer_id.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, '97-employer_id.cdb'))
df97_id = df97_id.rename(columns = rename_dict)
df97_id = clean_97(df97_id)
df97_id = df97_id.replace([-1, -2, -3, -4,-5,], np.nan)
drop_list = [c for c in df97_id if "_M_" in c or '_Y_' in c]
df97_id.drop(columns=drop_list,inplace=True)

uid_pattern = re.compile(r"^UID\.(\d{2})_(\d{4})$")
parts = []
for uid_col in df97_id.columns:
    match = uid_pattern.fullmatch(uid_col)
    if match is None:
        continue
    job_suffix, year = match.groups()
    start_col = f"STARTDATE.{job_suffix}_{year}"
    uid_values = pd.to_numeric(df97_id[uid_col],errors="coerce")
    valid = uid_values.notna()
    if not valid.any():
        continue
    if start_col in df97_id.columns:
        start_values = pd.to_numeric(df97_id[start_col],errors="coerce")
    else:
        start_values = pd.Series(pd.NA,index=df97_id.index,dtype="Int64")
    parts.append(pd.DataFrame({
            "PUBID": df97_id.loc[valid, "PUBID"].to_numpy(), "year": int(year), "job_suffix": job_suffix,
            "uid": uid_values.loc[valid].astype("Int64").to_numpy(),
            "startdate": start_values.loc[valid].astype("Int64").to_numpy(),}))
    
jobs_long = pd.concat(parts,ignore_index=True).drop_duplicates(["PUBID", "year", "job_suffix", "uid"])
uid_order = jobs_long.groupby(["PUBID", "uid"], as_index=False).agg(first_startdate=("startdate", "min"),first_seen_year=("year", "min"))
uid_order = uid_order.sort_values(["PUBID","first_startdate","first_seen_year","uid"],na_position="last",kind="stable")
uid_order["job_number"] = uid_order.groupby("PUBID").cumcount() + 1
matched = jobs_long.merge(uid_order[["PUBID", "uid", "job_number"]],on=["PUBID", "uid"],how="left",validate="many_to_one")

df97_id_match = matched[["PUBID", "year", "job_number", "job_suffix"]].drop_duplicates().sort_values(["PUBID", "year", "job_number"]).reset_index(drop=True)

In [63]:
df97_id_match

,PUBID,year,job_number,job_suffix
0,1,1997,1,01
1,1,1998,1,01
2,1,1999,1,02
3,1,1999,2,01
4,1,2000,2,01
...,...,...,...,...
210974,9022,2015,20,01
210975,9022,2017,20,01
210976,9022,2019,20,01
210977,9022,2021,20,01
